# 09 · SMOTE 오버샘플링 실험

> 불균형 대응 방법 중 '합성 오버샘플링(SMOTE)'을 직접 적용해 본다.
> SMOTE는 소수 클래스(Yes) 점들 사이를 보간해 **가짜 소수 샘플**을 만들어 비율을 맞춘다.
> 비교 대상: XGBoost(가중치) 0.244, MLP_focal 약 0.19.
> 핵심 주의: SMOTE는 **반드시 train에만** 적용한다 (val/test에 쓰면 누수).

## 0. 환경 설정
- imbalanced-learn(imblearn): SMOTE 등 불균형 리샘플링 도구 모음.
- 02에서 만든 데이터를 그대로 사용.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import xgboost as xgb
from imblearn.over_sampling import SMOTE

from utils import set_seed, compute_metrics, log_result, load_processed
set_seed(42)

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\딥러닝응용\TermProject")
OUT_DIR     = PROJECT_ROOT / "processed"
RESULTS_CSV = PROJECT_ROOT / "notebooks" / "results.csv"

train_df, val_df, test_df = load_processed(OUT_DIR)
TARGET = "went_on_backorder"
feature_cols = [c for c in train_df.columns if c != TARGET]
X_train = train_df[feature_cols].values.astype("float32")
y_train = train_df[TARGET].values.astype("int")
X_val   = val_df[feature_cols].values.astype("float32")
y_val   = val_df[TARGET].values.astype("int")
print("로드 완료. 피처:", len(feature_cols))

로드 완료. 피처: 33


## 1. SMOTE 적용 (train만)
- sampling_strategy=0.1: 소수 클래스를 다수의 10%까지 합성.
- k_neighbors=5: 가까운 소수 샘플 5개 사이를 보간해 가짜 점 생성.
val/test는 절대 건드리지 않는다.

In [2]:
print("SMOTE 전 분포 [No, Yes]:", np.bincount(y_train))
sm = SMOTE(sampling_strategy=0.1, k_neighbors=5, random_state=42)
X_smote, y_smote = sm.fit_resample(X_train, y_train)
print("SMOTE 후 분포 [No, Yes]:", np.bincount(y_smote), " (양성 {:.1%})".format(y_smote.mean()))

SMOTE 전 분포 [No, Yes]: [1341254    9034]


SMOTE 후 분포 [No, Yes]: [1341254  134125]  (양성 9.1%)


## 2. XGBoost on SMOTE
데이터가 이미 균형이라 scale_pos_weight는 쓰지 않는다(이중 보정 방지).

In [3]:
xgb_model = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                              random_state=42, n_jobs=-1, eval_metric="logloss")
xgb_model.fit(X_smote, y_smote)
xgb_prob = xgb_model.predict_proba(X_val)[:, 1]
xgb_m = compute_metrics(y_val, xgb_prob)
log_result("XGB_SMOTE", xgb_m, path=str(RESULTS_CSV))
print("XGB_SMOTE", xgb_m)

XGB_SMOTE {'PR_AUC': 0.2127, 'ROC_AUC': 0.951, 'Recall': 0.3878, 'Precision': 0.2272, 'F1': 0.2866, 'threshold': 0.5}


## 3. MLP on SMOTE
균형 데이터라 평범한 BCE를 쓴다(Focal/가중치 없이 SMOTE 효과만 본다).

In [4]:
def make_mlp(d):
    return nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

Xs = X_smote.astype("float32")
ys = y_smote.astype("float32")
loader = DataLoader(TensorDataset(torch.from_numpy(Xs), torch.from_numpy(ys)),
                    batch_size=4096, shuffle=True, drop_last=True)

set_seed(42)
mlp = make_mlp(len(feature_cols))
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
for epoch in range(8):
    mlp.train()
    for xb, yb in loader:
        opt.zero_grad()
        loss = loss_fn(mlp(xb).squeeze(1), yb)
        loss.backward()
        opt.step()

mlp.eval()
with torch.no_grad():
    mlp_prob = torch.sigmoid(mlp(torch.from_numpy(X_val)).squeeze(1)).numpy()
mlp_m = compute_metrics(y_val, mlp_prob)
log_result("MLP_SMOTE", mlp_m, path=str(RESULTS_CSV))
print("MLP_SMOTE", mlp_m)

MLP_SMOTE {'PR_AUC': 0.1885, 'ROC_AUC': 0.9473, 'Recall': 0.5091, 'Precision': 0.1679, 'F1': 0.2526, 'threshold': 0.5}


## 4. 결과 비교

In [5]:
import pandas as pd
res = pd.read_csv(RESULTS_CSV).drop_duplicates(subset="model", keep="last")
res = res.sort_values("PR_AUC", ascending=False)
res[["model", "PR_AUC", "ROC_AUC", "Recall", "Precision", "F1"]]

,model,PR_AUC,ROC_AUC,Recall,Precision,F1
7,XGBoost,0.2438,0.9618,0.8836,0.0598,0.1120
6,LightGBM,0.2237,0.9580,0.8907,0.0557,0.1048
14,XGB_SMOTE,0.2127,0.9510,0.3878,0.2272,0.2866
2,MLP_3_focal,0.1927,0.9463,0.1983,0.3109,0.2422
15,MLP_SMOTE,0.1885,0.9473,0.5091,0.1679,0.2526
3,MLP_4_focal_dropout,0.1859,0.9421,0.1301,0.3387,0.1880
0,MLP_1_plain_BCE,0.1859,0.9411,0.0217,0.4298,0.0413
1,MLP_2_weighted,0.1662,0.9469,0.8907,0.0451,0.0859
4,MLP_5_focal_drop_bn,0.1638,0.9443,0.0044,0.2041,0.0087
8,FT_Transformer,0.1446,0.9338,0.0713,0.2800,0.1136


---
### 결론
- SMOTE는 기존 방법을 넘지 못했다. 특히 **트리(XGBoost)는 오히려 떨어졌다** (약 0.244 -> 0.21).
- MLP도 Focal(~0.19)과 비슷하거나 약간 아래(~0.19)로, 이득이 없었다.
- 이유: 이진 플래그·파생변수에서 보간이 비현실적인 가짜 점을 만들어 경계를 흐리고, 트리는 이미 scale_pos_weight로 불균형을 잘 처리한다.
- 교훈(불균형 기법 정리): **손실레벨(Focal, 가중치) >= 오버샘플링(SMOTE) > 언더샘플링**. 강한 학습기엔 SMOTE 효과가 적다는 문헌과 일치.

이 노트북은 results.csv에 XGB_SMOTE, MLP_SMOTE 행을 추가한다. Run All로 직접 재현할 수 있다.